[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/notebooks/02_Working_with_Data_in_Python/02_pandas.ipynb)

# Multi-omics Data Science

# Introduction to Pandas (Python Data Analysis Library)

Pandas is a library that lets you work with many different kinds of data:

- Tabular data with columns of different types, like an Excel spreadsheet
- Time series data
- Any kind of observational or statistical data

In this notebook we learn pandas on **the dataset we will use for the rest of the course**: serum samples from 45 septic patients, 15 per group:

| group | meaning |
| :--- | :--- |
| `Con` | sepsis with **negative** blood cultures |
| `CSKP` | carbapenem-**susceptible** *Klebsiella pneumoniae* sepsis |
| `CRKP` | carbapenem-**resistant** *Klebsiella pneumoniae* sepsis |

For each patient we have clinical metadata, a proteomics matrix of 1458 protein groups (diaPASEF / DIA-NN) and a metabolomics matrix of 1073 named metabolites (targeted MRM).

Two things about the sample names are worth knowing before we touch the data. First, the sample identifiers of the carbapenem-susceptible group start with `KP` (`KP1` ... `KP15`) even though the group is called `CSKP` in the metadata; this inconsistency comes from the original study and we simply have to live with it. Second, the columns named `QC_pool*` (proteomics) or `QC*` (metabolomics) are **pooled quality-control injections, not patients**, so they must be excluded whenever we compute anything per patient.

## Objectives

- The concept of a table as a data frame
- How a table is organised: index, columns
- The data types a data frame can hold
- Reading/writing data from/to a file
- Selecting columns and rows (`loc`, `iloc`, boolean masks)
- Modifying a table, reshaping between wide and long format (`melt`, `pivot`) and combining tables (`merge`)
- Computing statistics and grouping (`groupby`, `value_counts`, `sort_values`)
- Handling missing values

In [ ]:
import pandas as pd

# All course data sets live in the course GitHub repository. We build the URLs
# once here and reuse them throughout the notebook.
COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"

# Clinical metadata, one row per patient
METADATA_URL = f"{BASE_URL}/metadata/sample_metadata.tsv"
# Omics matrices, one row per feature and one column per sample
PROTEINS_URL = f"{BASE_URL}/proteomics/data/protein_groups_matrix.tsv"
METABOLITES_URL = f"{BASE_URL}/metabolomics/data/metabolite_matrix.tsv"
# Annotation and published results (used in the exercises and in later notebooks)
METABOLITE_ANNOTATION_URL = f"{BASE_URL}/metabolomics/data/metabolite_annotation.tsv"
PUBLISHED_DEPS_URL = f"{BASE_URL}/proteomics/data/published_deps.tsv"

## Creating a data frame

In [ ]:
# A tiny table by hand: one row per patient, one column per variable
df = pd.DataFrame({
    'c_reactive_protein': [1.0, 3.0, 96.0, 145.0],   # mg/L
    'age': [55, 30, 59, 71],                         # years
    'group': ["Con", "Con", "CSKP", "CRKP"]
})

In [ ]:
df

In [ ]:
# The same table from a list of rows (a list of lists)
data = [
    [1.0, 55, "Con"],
    [3.0, 30, "Con"],
    [96.0, 59, "CSKP"],
    [145.0, 71, "CRKP"]
]

df = pd.DataFrame(data, columns=['c_reactive_protein', 'age', 'group'])
df

### From a `list` of `dictionaries`

In [ ]:
data = [
    {'c_reactive_protein': 1.0, 'age': 55, 'group': "Con"},
    {'c_reactive_protein': 3.0, 'age': 30, 'group': "Con"},
    {'c_reactive_protein': 96.0, 'age': 59, 'group': "CSKP"},
    {'c_reactive_protein': 145.0, 'age': 71, 'group': "CRKP"}
]

df = pd.DataFrame(data)
df

### Creating an empty `DataFrame`

In [ ]:
df = pd.DataFrame()
df['c_reactive_protein'] = [1.0, 3.0, 96.0, 145.0]
df['age'] = [55, 30, 59, 71]
df['group'] = ["Con", "Con", "CSKP", "CRKP"]

df

### Exercise

Create the following DataFrame. It holds the median value of two clinical markers in two of the patient groups (one row per group/marker combination; we will come back to this "long" table shape later).

|  | Group | Marker | Median |
| ---| :--: | :----:  | :--: |
| 0  | Con  | procalcitonin       | 0.07 |
| 1  | CRKP | procalcitonin       | 2.33 |
| 2  | Con  | c_reactive_protein  | 5.00 |
| 3  | CRKP | c_reactive_protein  | 105.00 |

# Reading/Writing Data from/to a File


Pandas has several functions to read data in many formats. Most of them start with `read_`:

| File Type | Function Name |
| :----:    |  :---:  |
| Excel | `pd.read_excel` |
| CSV, TSV | `pd.read_csv` |
| H5, HDF, HDF5 | `pd.read_hdf` |
| JSON  | `pd.read_json` |
| SQL | `pd.read_sql_table` |


### Reading Data

We will read the data directly from GitHub. To get such a link yourself, go to the file on GitHub, click on it, and use the **Raw** option. That opens the file as plain text and gives you the direct URL, which `pandas` can read without any further work.

The course data are in this repository:

https://github.com/Multiomics-Analytics-Group/course_multi-omics_analysis

and we already assembled the URLs in the cell at the top of the notebook, e.g. `METADATA_URL` points to `metadata/sample_metadata.tsv`.

All course files are **tab-separated**, so we must pass `sep='\t'` to `pd.read_csv`.

**Comma separated/Tab separated files (.csv, .tsv)**

In [ ]:
metadata = pd.read_csv(METADATA_URL, sep='\t')
metadata.head()

**Excel files (.xls, .xlsx)**

In [ ]:
# The same table stored as a spreadsheet would be read like this:
# metadata = pd.read_excel("sample_metadata.xlsx", sheet_name='metadata')

### Writing Data

**Comma separated/Tab separated files (.csv, .tsv)**

In [ ]:
# Write the table back out as a tab-separated file.
# index=False leaves out the row numbers, which are not part of the data.
metadata.to_csv("sample_metadata_copy.tsv", sep='\t', index=False)

**Excel files (.xls, .xlsx)**

In [ ]:
# metadata.to_excel("sample_metadata.xlsx", sheet_name='metadata', index=False)

## Accessing data in Google Drive

You can use your Google Drive space to read and store data. To do so you first have to mount your Google Drive in Colab, following these steps:

1. Run the code below. It gives you an authentication link so that you can access your Google Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

2. Open the link

3. Choose your Google account

4. Allow Google Drive Stream to access your Google account

5. Copy the code it returns and paste it into a text box like the one shown below.

![image.png](https://i0.wp.com/neptune.ai/wp-content/uploads/2022/10/colab-code-copy.png?ssl=1)

Once mounted, you can browse your files in the file panel on the left and read them exactly as shown above:
![image.png](https://i0.wp.com/neptune.ai/wp-content/uploads/2022/10/colab-drive.png?ssl=1)

# Indexing

In [ ]:
metadata.index

In [ ]:
# The row numbers 0...44 carry no meaning. The sample identifier does,
# so we can use it as the index of the table.
metadata_by_sample = metadata.set_index("sample_id")
metadata_by_sample.head()

In [ ]:
# .loc selects by label, .iloc selects by position
print("CRP of patient CRKP1:", metadata_by_sample.loc["CRKP1", "c_reactive_protein"])
print("First row, first column:", metadata_by_sample.iloc[0, 0])

# Both accept lists, so we can pick several rows and columns at once
metadata_by_sample.loc[["Con1", "KP1", "CRKP1"], ["group", "age", "c_reactive_protein"]]

### Structural Properties

In [ ]:
metadata.shape

In [ ]:
metadata.dtypes

In [ ]:
metadata.index

In [ ]:
metadata.columns

In [ ]:
metadata['c_reactive_protein']

### Inspecting Tables

In [ ]:
metadata.head()

In [ ]:
metadata.tail(3)

In [ ]:
# info() summarises the columns, their types and how many values are not missing
metadata.info()

# describe() summarises the numeric columns
metadata[['age', 'bmi', 'c_reactive_protein', 'procalcitonin']].describe()

### Selection/Filtering

#### Selection
To select or filter we can use conditions, as we saw with booleans:

```python
metadata['c_reactive_protein'] > 100   # returns a series of `True` where the value is above 100 and `False` otherwise
```

To filter, simply put the condition in square brackets and only the `True` rows are kept:

```python
selected_rows = metadata[metadata['c_reactive_protein'] > 100]
```
or
```python
selected_rows = metadata.loc[metadata['c_reactive_protein'] > 100]
```

In [ ]:
# A boolean mask: True for every carbapenem-resistant K. pneumoniae patient
mask = metadata['group'] == "CRKP"
print(mask.head())

crkp = metadata[mask]
print("CRKP patients:", crkp.shape)

# Masks can be combined with & (and) and | (or).
# Each condition needs its own parentheses.
severe_infected = metadata[(metadata['c_reactive_protein'] > 100) & (metadata['group'] != "Con")]
severe_infected[['sample_id', 'group', 'age', 'c_reactive_protein', 'procalcitonin']]

# Modifying a Table

### Removing a column

In [ ]:
clinical = metadata.copy()    # work on a copy so the original table stays intact
del clinical['group_order']   # a helper column used only for plotting order

In [ ]:
clinical.head()

### Creating a new column

In [ ]:
# The comorbidities are stored as 0/1 flags, one column each.
# Summing them along the rows (axis=1) gives the number of comorbidities per patient.
comorbidities = ['diabetes', 'heart_disease', 'copd',
                 'liver_disease', 'cerebrovascular_disease', 'kidney_disease']

clinical['n_comorbidities'] = clinical[comorbidities].sum(axis=1)
clinical[['sample_id', 'group'] + comorbidities + ['n_comorbidities']].head()

### Wide and long format: `melt` and `pivot`

Omics data usually arrive in **wide** format: one row per feature (protein, metabolite) and one column per sample. That is compact and convenient for matrix operations, but most plotting and statistics functions expect **long** format: one row per measurement, with the sample and the feature given as columns.

`melt` goes from wide to long, `pivot` goes back from long to wide.

In [ ]:
proteins = pd.read_csv(PROTEINS_URL, sep='\t')
print("protein matrix:", proteins.shape)
proteins.iloc[:5, :8]

The first four columns describe the protein group, the remaining columns are samples. Note the last three: `QC_pool1`-`QC_pool3` are pooled quality-control injections, **not patients**, so we keep them apart from the patient columns.

In [ ]:
info_columns = ['protein_group', 'protein_names', 'genes', 'description']
qc_columns = [c for c in proteins.columns if c.startswith("QC")]
sample_columns = [c for c in proteins.columns
                  if c not in info_columns and c not in qc_columns]

print("annotation columns:", info_columns)
print("QC columns:", qc_columns)
print("patient columns:", len(sample_columns), "->", sample_columns[:3], "...", sample_columns[-3:])

In [ ]:
# Wide -> long
proteins_long = proteins.melt(
    id_vars=['protein_group', 'genes'],
    value_vars=sample_columns,
    var_name='sample_id',
    value_name='intensity',
)
print(proteins_long.shape, "= 1458 proteins x 45 patients")
proteins_long.head()

In [ ]:
# Long -> wide again
proteins_wide = proteins_long.pivot(index='protein_group',
                                    columns='sample_id',
                                    values='intensity')
print(proteins_wide.shape)
proteins_wide.iloc[:5, :5]

### Combining tables: `merge`

The long table knows the sample identifier, but not which group the patient belongs to. That information is in `sample_metadata.tsv`, and `merge` joins the two tables on the shared column `sample_id` (the same idea as a `JOIN` in SQL).

The same operation works for features instead of samples: the metabolomics matrix (`METABOLITES_URL`) can be merged with its annotation table (`METABOLITE_ANNOTATION_URL`) on the column `metabolite` to attach the compound class and the KEGG/HMDB identifiers. Only 408 of the 1073 measured metabolites have an annotation, so there `how` matters: `how='left'` keeps all metabolites and leaves the annotation columns empty for the rest, while `how='inner'` keeps only the annotated ones.

In [ ]:
proteins_annotated = proteins_long.merge(
    metadata[['sample_id', 'group', 'group_label', 'age', 'sex']],
    on='sample_id',
    how='left',      # keep every row of the left table
)
print(proteins_annotated.shape)
print("rows without a matching sample:", proteins_annotated['group'].isna().sum())
proteins_annotated.head()

# Statistics

```python
metadata.describe()                     # describes the numeric columns
metadata['age'].describe()              # describes one particular column
metadata['bmi'].count()                 # counts the non-missing values
metadata['group'].nunique()             # counts the unique values
metadata['group'].unique()              # returns the unique values of the column
metadata['group'].value_counts()        # returns the unique values and how often each occurs

metadata['c_reactive_protein'].max()
metadata['c_reactive_protein'].mean()
metadata['c_reactive_protein'][metadata['group'] == 'CRKP'].sum()
```


#### Computing the mean of a column

In [ ]:
metadata['c_reactive_protein'].mean()

#### Computing the sum of a column

In [ ]:
# The comorbidity flags are 0/1, so the sum is the number of patients with diabetes
metadata['diabetes'].sum()

In [ ]:
metadata['age'].min()

In [ ]:
metadata['age'].max()

In [ ]:
# How many patients are there per group?
print(metadata['group'].value_counts())

# sort_values orders the rows; here the patients with the highest CRP come first
print(metadata.sort_values('c_reactive_protein', ascending=False)
              [['sample_id', 'group', 'c_reactive_protein']].head())

# groupby splits the table by group and computes the statistic within each group
metadata.groupby('group')[['age', 'bmi', 'c_reactive_protein', 'procalcitonin']].mean()

# Exercise

Using the proteomics matrix and the sample metadata of the course

```python
proteins = pd.read_csv(PROTEINS_URL, sep='\t')
metadata = pd.read_csv(METADATA_URL, sep='\t')
```

answer the following questions.

1. Show the first and the last 5 lines of the protein matrix

2. Create a new table that contains only the columns: protein_group, genes and the 15 `CRKP` samples

3. Create a new long-format table with: protein_group, genes, sample_id, intensity and group, for the infected patients only (`CSKP` and `CRKP`)


4. Which patient has the highest C-reactive protein value? And which protein group has the highest mean intensity across patients?

5. How many patients are there per group, and for how many protein groups is a gene name reported?

# Missing Values

| method | description
| ---:  | :---- |
**`isna()`** | returns True for every NaN |
**`notna()`** | returns False for every NaN |
**`dropna()`** | returns only the rows that contain no NaNs |


If the data frame contains missing values and we need a complete matrix, we can use the following methods to impute values:

| method | description |
| ----: |  :---- |
| **`fillna()`** | replaces NaNs with a given value |
| **`ffill()`** | replaces NaNs with the previous non-NaN value |
| **`bfill()`** | replaces NaNs with the next non-NaN value |
| **`interpolate()`** | interpolates between the previous and the following values |

Missing values are not an academic problem in proteomics: about **27 %** of the protein matrix is missing. A value can be missing because the protein is genuinely absent (or below the detection limit) in that patient, or simply because the instrument did not pick it up in that run. The two cases call for different treatments, so always look at *where* the missing values are before imputing anything.

In [ ]:
protein_values = proteins[sample_columns]

print("Missing values in total:", int(protein_values.isna().sum().sum()),
      "out of", protein_values.size)
print("Fraction missing: {:.1%}".format(protein_values.isna().to_numpy().mean()))

# How many patients is each protein missing in?
missing_per_protein = protein_values.isna().sum(axis=1)
missing_per_protein.value_counts().sort_index().head(10)

In [ ]:
# The rows in which the protein was not quantified in patient Con1
proteins.loc[proteins['Con1'].isna(),
             ['protein_group', 'genes', 'Con1', 'Con2', 'Con3']].head()

In [ ]:
# Option 1: keep only the proteins quantified in every patient ("complete cases")
complete = proteins.dropna(subset=sample_columns)
print("complete proteins:", complete.shape[0], "of", proteins.shape[0])

# Option 2: keep the proteins quantified in at least 70 % of the patients
keep = protein_values.notna().mean(axis=1) >= 0.7
print("proteins in >=70 % of patients:", int(keep.sum()))

# Option 3: impute, here with the lowest intensity observed anywhere in the matrix
# (a crude stand-in for "below the detection limit" - we discuss better options later)
imputed = proteins.copy()
imputed[sample_columns] = imputed[sample_columns].fillna(protein_values.min().min())
print("missing values after imputation:", int(imputed[sample_columns].isna().sum().sum()))

# Exercise

1. Using the protein matrix, compute the number of missing values per patient and per protein group

```python
proteins = pd.read_csv(PROTEINS_URL, sep='\t')
sample_columns = [c for c in proteins.columns
                  if c not in ['protein_group', 'protein_names', 'genes', 'description']
                  and not c.startswith("QC")]
```

2. Assign the value **0** to the missing values in the column **Con1**. Why is 0 a poor choice for an intensity?

3. Replace the missing values of each protein group with the median intensity of that protein across the patients (hint: `.median(axis=1)` together with `fillna`), and compare how many protein groups you keep with this approach versus dropping every protein group that has any missing value.

## Further Resources

- Python Pandas: https://pandas.pydata.org/pandas-docs/stable/
- Course repository: https://github.com/Multiomics-Analytics-Group/course_multi-omics_analysis
